# Load pre-trained weights and run models

In this tutorial, we will show how to load the pre-trained Tadpole weights and run the models.
To help users quickly get started with Tadpole, we provide it as a simple PyTorch Module that can be integrated into any training or inference pipeline without modifying code.


We strongly recommend users to read the [paper](https://arxiv.org/abs/2408.11104) before using the code.

## Tadpole Autoencoder

The Tadpole autoencoder is used for pre-training and downstream autoencoding tasks. It takes a 3D input with shape of (B, C, X, Y, Z) and returns the reconstructed output with the same shape. A simple example is shown below.


In [8]:
from tadpole.model.autoencoder import TadpoleAutoencoder
import torch
DEVICE="cuda" if torch.cuda.is_available() else "cpu"

ae=TadpoleAutoencoder(size="B")

inputs = torch.randn(1, 3, 128, 128, 128, device=DEVICE)
ae.to(DEVICE)
print(ae(inputs).shape)

torch.Size([1, 3, 128, 128, 128])


`TadpoleAutoencoder` class takes the following arguments:
* size (Literal["S", "B", "L"]): Size of the model, one of "S", "B", or "L".

* weight_encoder/weight_decoder (Optional[Union[str,Dict,OrderedDict]]): Path to encoder weights or state dict for encoder. If None, the encoder will be randomly initialized. Default is None.

The pre-trained weights of Tadpole are available at [Hugging Face](https://huggingface.co/thuerey-group/Tadpole). You can either download the weights and load them from a local path, or directly load the weights through the Hugging Face API. For example, to load the "B" size model, you can simply do:

In [ ]:
from huggingface_hub import hf_hub_download
ae=TadpoleAutoencoder(
    size="B",
    weight_encoder=hf_hub_download(repo_id="thuerey-group/Tadpole",filename="tadpole_b_encoder.safetensors"),
    weight_decoder=hf_hub_download(repo_id="thuerey-group/Tadpole",filename="tadpole_b_decoder.safetensors"),
    )

* encoder/decoder_ft_state (Union[Literal["frozen","FPFT"],int]): Fine-tuning state for encoder/decoder. Can be a positive integer indicating the rank for LoRA fine-tuning, "frozen" to freeze the encoder weights, or "FPFT" to enable full-parameter fine-tuning. Default is "FPFT".

Below, we show an example setting different fine-tuning states for the encoder and decoder.

In [3]:
def num_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6 

ae_full=TadpoleAutoencoder(size="B")
ae_lora_32=TadpoleAutoencoder(size="B", encoder_ft_state=32, decoder_ft_state=32)
ae_frozen=TadpoleAutoencoder(size="B", encoder_ft_state="frozen", decoder_ft_state="frozen")
print(f"Full fine-tuning: {num_trainable_params(ae_full):.2f}M trainable parameters")
print(f"LoRA fine-tuning with rank 32: {num_trainable_params(ae_lora_32):.2f}M trainable parameters")
print(f"Fully frozen: {num_trainable_params(ae_frozen):.2f}M trainable parameters")

Full fine-tuning: 38.07M trainable parameters
LoRA fine-tuning with rank 32: 2.81M trainable parameters
Fully frozen: 0.00M trainable parameters


* latent_type (Literal["sample", "mode"]): How to sample from the latent distribution, either "sample" or "mode". Default is "sample".

The arguments above are used to control the type of latent space generated by the encoder. Since Tadpole is pre-trained with a VAE objective, the encoder generates a distribution in the latent space. By default, the model samples from this distribution to obtain the latent representation, enabling more diverse outputs. However, users can also choose to use the distribution's mode (i.e., the mean) as the latent representation, which can lead to more deterministic outputs.

* encoder_crop_size (int): Size to crop input for encoder. If None, no cropping will be applied, and the entire input will be processed as a single crop. Default is 64.

The Tadpole autoencoder processes the input in single-channel crops; the arguments above control the crop size. For example, if the input has a shape of (B, C, 128, 128, 128) and the encoder_crop_size is set to 64, the actual data shape processed by the encoder will be (B $\times$ C $\times$ 8, 1, 64, 64, 64), where 8 is the number of crops for each channel. If the encoder_crop_size is set to None, the data shape processed by the encoder will be (B $\times$ C, 1, 128, 128, 128), and no cropping will be applied.

* max_internal_batchsize (Optional[int]): Maximum batch size for internal processing. If None, all crops will be processed in a single batch. Default is None.

As mentioned above, the Tadpole autoencoder processes the input in single-channel crops. The max_internal_batchsize controls the batch size for processing these crops. For example, if the input has a shape of (B, C, 128, 128, 128) and the encoder_crop_size is set to 64, there will be B $\times$ C $\times$ 8 crops with a shape of (1, 64, 64, 64). If the max_internal_batchsize is set to 16, these crops will be processed in B $\times$ C $\times$ 8/16 batches with a shape of (16, 1, 64, 64, 64) in each batch. If the max_internal_batchsize is set to None, all crops will be processed in a single batch with a shape of (B $\times$ C $\times$ 8, 1, 64, 64, 64).

## Tadpole DFT
The Tadpole DFT is used in the fine-tuning of dynamics tasks. It takes a 3D input with shape of (B, C, X, Y, Z) and returns the predicted next frame with the same shape. A simple example is shown below.

In [7]:
from tadpole.model.dft import TadpoleDFT
dft=TadpoleDFT(size="B")
inputs = torch.randn(1, 3, 128, 128, 128,device=DEVICE)
dft.to(DEVICE)
print(dft(inputs).shape)

torch.Size([1, 3, 128, 128, 128])


The `TadpoleDFT` class also has a lot of arguments. Besides the same arguments as `TadpoleAutoencoder`, it also takes the following arguments:

* input_channels (int): Number of input channels. Default is 3.

* subnetwork (Union[Literal["default"], None, Module]): Subnetwork to apply in the latent space. Can be "default" to use a default subnetwork based on the model size. If None, no subnetwork will be applied. If a torch.nn.Module is provided, it will be used as the subnetwork. Default is "default".

The default `subnetwork` is a simple encoder-only transformer as discussed in the paper. You could provide any subnetwork you prefer, as long as it takes the correct input and output shapes. Below, we list the input and output shapes for the subnetwork:

| Size | Input/output shape (latent shape)                  |
|------|--------------------------------------|
| S    | [B, C $\times$ 256, X/16,Y/16,Z/16]  |
| B    | [B, C $\times$ 512, X/16,Y/16,Z/16]  |
| L    | [B, C $\times$ 1024, X/16,Y/16,Z/16] |


* encoder/decoder/subnetwork_activation_ckpt (bool): Whether to apply activation checkpointing to the encoder/decoder/subnetwork_activation_ckpt. Default is False.

This argument allows the model to use [activation checkpointing](https://pytorch.org/blog/activation-checkpointing-techniques/) to save GPU memory at the cost of extra computation. Usually, enabling only the activation checkpointing for the encoder is enough to save a lot of memory.

## Tadpole Encoder

Tadpole encoder class is used to only encode the input into the latent space, which can be used in the latent generative model downstream tasks. It takes a 3D input with shape of (B, C, X, Y, Z) and return the latent representation with the shape listed in the above table. A simple example is shown below.

In [ ]:
from tadpole.model.encoder import TadpoleEncoder

encoder=TadpoleEncoder(
    size="B", 
    )
inputs = torch.randn(1, 3, 128, 128, 128,device=DEVICE)
encoder.to(DEVICE)
print(encoder(inputs).shape)

torch.Size([1, 1536, 8, 8, 8])


Tadpole encoder also has some arguments, which are the same as `TadpoleAutoencoder`. Users can refer to the above section for the detailed description of these arguments.